<a href="https://colab.research.google.com/github/alexxvaz/SenacMinereacaoDadosEleicao2026/blob/main/00_preparacao_dados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fase 0 — Preparação de Dados: Candidatos 2026 (TSE)

Este notebook baixa e prepara os dados de candidatos das Eleições 2026, e salva um **dataset limpo** que todas as fases seguintes (1 a 4) vão reutilizar — assim ninguém precisa baixar e limpar os dados de novo em cada notebook.

## Configuração

Ajuste as três variáveis abaixo antes de rodar. É a **única coisa que muda** entre a análise do professor (Brasil, Governador) e o trabalho de vocês (um cargo, uma UF).

In [7]:
CARGO = "SENADOR"   # ex.: "GOVERNADOR", "SENADOR", "DEPUTADO FEDERAL"
UF = None               # None = Brasil inteiro; ou a sigla da UF, ex.: "SP", "BA", "MG"
ANO_ELEICAO = 2026

> **Para os alunos:** troquem `CARGO` para `"SENADOR"` ou `"DEPUTADO FEDERAL"` e `UF` para a sigla do estado do seu trabalho (ex.: `UF = "PE"`). O resto do notebook não muda.

> **Sobre "congelar" a versão dos dados:** o arquivo do TSE pode ser atualizado por eles a qualquer momento (novas candidaturas, correções, impugnações). Para não ficar refém disso, os zips baixados manualmente ficam parados em `dados/` e só mudam quando você baixar uma versão nova de propósito. O que fica **versionado no git** é o resultado desta fase (`dados/*.csv`) — depois de rodar e conferir, faça `git commit` desse CSV: esse commit *é* a versão congelada que a turma toda vai usar.

## 1) Preparação do ambiente

In [8]:
import warnings
warnings.filterwarnings("ignore")

import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
pd.set_option('display.max_columns', None)

os.makedirs('dados', exist_ok=True)

## 2) Obter os dados

O download automático pelo CDN do TSE é bloqueado por um firewall (Akamai) que rejeita requisições que não vêm de um navegador de verdade — isso acontece tanto no Colab quanto localmente, então não vale a pena automatizar. **O fluxo é manual:**

1. Baixe os dois arquivos pelo navegador:
   - Candidatos: `https://cdn.tse.jus.br/estatistica/sead/odsele/consulta_cand/consulta_cand_2026.zip`
   - Bens: `https://cdn.tse.jus.br/estatistica/sead/odsele/bem_candidato/bem_candidato_2026.zip`
2. Salve os dois **dentro da pasta `dados/`**, com esses nomes exatos:
   - `dados/consulta_cand_2026.zip`
   - `dados/bem_candidato_2026.zip`
3. Rode as células abaixo — elas conferem se os arquivos estão no lugar certo e descompactam.

In [9]:
ZIP_CAND = f"dados/consulta_cand_{ANO_ELEICAO}.zip"
ZIP_BEM = f"dados/bem_candidato_{ANO_ELEICAO}.zip"

def conferir_zip(caminho):
    if not os.path.exists(caminho):
        raise FileNotFoundError(
            f"Não encontrei {caminho}. Baixe o arquivo pelo navegador e salve exatamente nesse caminho "
            "(veja os links na célula de markdown acima)."
        )
    if not zipfile.is_zipfile(caminho):
        raise ValueError(f"{caminho} existe, mas não é um .zip válido — baixe de novo pelo navegador.")
    tamanho_mb = os.path.getsize(caminho) / (1024 * 1024)
    print(f"OK: {caminho} ({tamanho_mb:.1f} MB)")

def descompactar(nome_arquivo_zip, destino):
    with zipfile.ZipFile(nome_arquivo_zip, 'r') as zip_ref:
        zip_ref.extractall(destino)
    print(f"Descompactado em {destino}")

conferir_zip(ZIP_CAND)
conferir_zip(ZIP_BEM)

OK: dados/consulta_cand_2026.zip (3.1 MB)
OK: dados/bem_candidato_2026.zip (3.6 MB)


In [10]:
descompactar(ZIP_CAND, './files_cand')
descompactar(ZIP_BEM, './files_bem')

Descompactado em ./files_cand
Descompactado em ./files_bem


## 3) Filtrar candidatos (CARGO + UF)

In [11]:
df = pd.read_csv(f'./files_cand/consulta_cand_{ANO_ELEICAO}_BRASIL.csv', sep=';', encoding='latin-1')

# Rastreabilidade: o próprio TSE grava quando gerou este extrato — assim dá pra saber
# exatamente qual snapshot da base está sendo usado, mesmo que o arquivo mude no futuro.
print(f"Extrato gerado pelo TSE em: {df['DT_GERACAO'].iloc[0]} {df['HH_GERACAO'].iloc[0]}")

print("\nCargos disponíveis:")
print(df.DS_CARGO.value_counts())

dfCargo = df[df['DS_CARGO'] == CARGO].copy()
if UF is not None:
    dfCargo = dfCargo[dfCargo['SG_UF'] == UF].copy()

# uma linha por candidato: fica com o turno mais avançado (2º turno quando existir)
dfCargoFinal = dfCargo.loc[dfCargo.groupby('SQ_CANDIDATO')['NR_TURNO'].idxmax()]

print()
print(f"CARGO={CARGO!r}  UF={UF!r}  ->  {dfCargoFinal.shape[0]} candidatos")
dfCargoFinal[['SQ_CANDIDATO', 'NM_CANDIDATO', 'SG_UF', 'SG_PARTIDO', 'DT_NASCIMENTO']].head()

Extrato gerado pelo TSE em: 21/09/2026 19:31:26

Cargos disponíveis:
DS_CARGO
DEPUTADO ESTADUAL     11293
DEPUTADO FEDERAL       7801
DEPUTADO DISTRITAL      433
2º SUPLENTE             350
1º SUPLENTE             349
SENADOR                 319
VICE-GOVERNADOR         211
GOVERNADOR              201
PRESIDENTE               14
VICE-PRESIDENTE          14
Name: count, dtype: int64

CARGO='SENADOR'  UF=None  ->  319 candidatos


,SQ_CANDIDATO,NM_CANDIDATO,SG_UF,SG_PARTIDO,DT_NASCIMENTO
7828,10002533895,CYLMARA FERNANDES DA ROCHA GRIPP,AC,REPUBLICANOS,04/09/1973
18212,10002535804,SERGIO DE OLIVEIRA CUNHA,AC,PSD,20/04/1960
2542,10002536441,JORGE NEY VIANA MACEDO NEVES,AC,PT,20/09/1959
5161,10002536710,RIBAMAR DE SOUSA FEITOZA JÚNIOR,AC,DC,09/05/1982
2540,10002544110,INACIO ALVES MOREIRA NETTO,AC,PSOL,14/05/1970


## 4) Patrimônio declarado (bens)

In [32]:
dfBem = pd.read_csv('./files_bem/bem_candidato_%d_BRASIL.csv' % ANO_ELEICAO, sep=';', encoding='latin-1')
dfBem['VR_BEM_CANDIDATO'] = dfBem['VR_BEM_CANDIDATO'].str.replace(',', '.').astype(float)

def categorize_assets(asset):
    a = asset.lower()
    if 'veículo' in a or 'moto' in a or 'aeronave' in a or 'embarcação' in a:
        return 'BENS MÓVEIS'
    elif 'casa' in a or 'terreno' in a or 'apartamento' in a or 'prédio' in a or 'loja' in a or 'terra nua' in a:
        return 'BENS IMÓVEIS'
    elif 'poupança' in a or 'renda fixa' in a or 'cdb' in a or 'rdb' in a:
        return 'INVESTIMENTOS RENDA FIXA'
    elif 'ações' in a or 'fundo' in a or 'investimento' in a or 'mercado' in a:
        return 'INVESTIMENTOS RENDA VARIÁVEL'
    elif 'dinheiro' in a or 'depósito bancário' in a or 'numerário' in a:
        return 'DINHEIRO'
    else:
        return 'OUTROS BENS'

dfBem['Categoria_BEM'] = dfBem['DS_TIPO_BEM_CANDIDATO'].apply(categorize_assets)

dfBem_pivot = dfBem.pivot_table(
    index='SQ_CANDIDATO', columns='Categoria_BEM', values='VR_BEM_CANDIDATO', aggfunc='sum'
)

dfBem_pivot.fillna(0, inplace=True)
dfBem_pivot['Total_Bens'] = dfBem_pivot.sum(axis=1)
dfBem_pivot['Total_Bens_Log'] = np.log1p(dfBem_pivot['Total_Bens'])

dfBem_pivot[['Total_Bens', 'Total_Bens_Log']].describe()


Categoria_BEM,BENS IMÓVEIS,BENS MÓVEIS,DINHEIRO,INVESTIMENTOS RENDA FIXA,INVESTIMENTOS RENDA VARIÁVEL,OUTROS BENS
SQ_CANDIDATO,,,,,,
10002532416,NaN,60000.0,NaN,NaN,NaN,NaN
10002532417,NaN,266330.0,NaN,50000.0,NaN,NaN
10002532418,3250000.0,NaN,NaN,NaN,NaN,NaN
10002532419,NaN,195000.0,NaN,NaN,NaN,3.920000e+06
10002532420,300000.0,155000.0,NaN,NaN,NaN,1.771245e+06
...,...,...,...,...,...,...
280002552485,874166.8,99000.0,NaN,NaN,NaN,1.250261e+06
280002552486,50000.0,60000.0,NaN,NaN,NaN,NaN
280002553883,NaN,NaN,NaN,NaN,NaN,4.952243e+08


"\ndfBem_pivot.fillna(0, inplace=True)\ndfBem_pivot['Total_Bens'] = dfBem_pivot.sum(axis=1)\ndfBem_pivot['Total_Bens_Log'] = np.log1p(dfBem_pivot['Total_Bens'])\n\ndfBem_pivot[['Total_Bens', 'Total_Bens_Log']].describe()\n"

## 5) Juntar tudo, calcular idade e limpar

A idade mínima elegível **depende do cargo** (regra constitucional) — por isso o filtro de idade é montado a partir de `CARGO`, não fixo.

In [13]:
IDADE_MINIMA_POR_CARGO = {
    'PRESIDENTE': 35, 'VICE-PRESIDENTE': 35,
    'GOVERNADOR': 30, 'VICE-GOVERNADOR': 30,
    'SENADOR': 35,
    'DEPUTADO FEDERAL': 21, 'DEPUTADO ESTADUAL': 21, 'DEPUTADO DISTRITAL': 21,
    'PREFEITO': 21, 'VICE-PREFEITO': 21,
    'VEREADOR': 18,
}
idade_minima = IDADE_MINIMA_POR_CARGO.get(CARGO, 18)
print(f"Idade mínima elegível para {CARGO}: {idade_minima} anos")

dfCandBem = dfCargoFinal.merge(dfBem_pivot, on='SQ_CANDIDATO', how='left')
dfCandBem['Total_Bens'] = dfCandBem['Total_Bens'].fillna(0)
dfCandBem['Total_Bens_Log'] = dfCandBem['Total_Bens_Log'].fillna(0)

dfCandBem['DT_ELEICAO'] = pd.to_datetime(dfCandBem['DT_ELEICAO'], format='%d/%m/%Y')
dfCandBem['DT_NASCIMENTO'] = pd.to_datetime(dfCandBem['DT_NASCIMENTO'], format='%d/%m/%Y')
dfCandBem['IDADE'] = (dfCandBem['DT_ELEICAO'] - dfCandBem['DT_NASCIMENTO']).dt.days // 365

# remover idades inválidas (erro de cadastro) usando a idade mínima constitucional do cargo
antes = dfCandBem.shape[0]
dfCandBem = dfCandBem[dfCandBem['IDADE'].between(idade_minima, 100)].copy()
print(f"Candidatos removidos por idade inválida: {antes - dfCandBem.shape[0]}")

print(dfCandBem.shape)
dfCandBem[['NM_CANDIDATO', 'SG_UF', 'IDADE', 'Total_Bens', 'Total_Bens_Log']].sample(min(5, len(dfCandBem)))

Idade mínima elegível para SENADOR: 35 anos
Candidatos removidos por idade inválida: 1
(318, 59)


,NM_CANDIDATO,SG_UF,IDADE,Total_Bens,Total_Bens_Log
263,MARIA TERESA SAENZ SURITA GUIMARÃES,RR,70.0,8481615.57,15.953412
90,ANDRE LUIZ CARVALHO RIBEIRO,MA,37.0,1829911.74,14.419779
248,SILVIA CRISTINA AMANCIO CHAGAS,RO,52.0,1372191.69,14.131921
235,MILTON BATISTA CARDOSO,RS,69.0,36200.00,10.496842
304,HENRIQUE ALVES DA ROCHA,SE,59.0,371500.00,12.825307


> **Nota:** propositalmente **não removemos outliers de patrimônio aqui** — a Fase 1 (análise descritiva) precisa vê-los para serem discutidos, e o tratamento de outliers é específico de cada técnica (vamos fazer isso dentro da Fase 2, clusterização).

## 6) Salvar o dataset limpo

In [14]:
nome_uf = UF if UF is not None else 'BRASIL'
nome_cargo = CARGO.replace(' ', '_')
arquivo_saida = f"dados/candidatos_{nome_cargo}_{nome_uf}_{ANO_ELEICAO}.csv"

dfCandBem.to_csv(arquivo_saida, index=False)
print(f"Dataset salvo em: {arquivo_saida}  ({dfCandBem.shape[0]} candidatos, {dfCandBem.shape[1]} colunas)")

Dataset salvo em: dados/candidatos_SENADOR_BRASIL_2026.csv  (318 candidatos, 59 colunas)


# Agregando os valores **Contratados**

In [38]:
df_despesas = pd.read_csv( 'dados/despesas_contratadas_candidatos_2026_BRASIL.csv', sep=';', encoding='latin-1' )
df_despesas['VR_DESPESA_CONTRATADA'] = df_despesas['VR_DESPESA_CONTRATADA'].str.replace(',', '.').astype(float)
dfDespesaAgg = df_despesas.pivot_table(
    index='SQ_CANDIDATO', values='VR_DESPESA_CONTRATADA', aggfunc='sum' )

dfDespesaAgg.fillna(0, inplace=True)
dfDespesaAgg['VR_DESPESA_CONTRATADA'] = dfDespesaAgg.sum(axis=1)
dfDespesaAgg['VR_DESPESA_CONTRATADA_log'] = np.log1p(dfDespesaAgg['VR_DESPESA_CONTRATADA'])

dfDespesaAgg[['VR_DESPESA_CONTRATADA', 'VR_DESPESA_CONTRATADA_log']].describe()


,VR_DESPESA_CONTRATADA,VR_DESPESA_CONTRATADA_log
count,1.904000e+04,19040.000000
mean,1.918697e+05,7.961311
std,8.421425e+05,5.101449
min,0.000000e+00,0.000000
25%,0.000000e+00,0.000000
50%,2.102390e+04,9.953463
75%,1.352095e+05,11.814588
max,6.567712e+07,18.000261


#Juntar os arquivos CSV

In [43]:
df_final = pd.merge( dfCandBem, dfDespesaAgg, on ='SQ_CANDIDATO', how ='inner' )
df_final.to_csv('dados/resultado_senado_juntos.csv', index=False)

---
## Congelando esta versão

Depois de conferir o resultado acima, faça o commit desse arquivo:

```bash
git add dados/*.csv
git commit -m "Dados: candidatos <CARGO> <UF> <ANO> (snapshot TSE de <DT_GERACAO>)"
```

A partir daí, **esse commit é a versão oficial** que a Fase 1 em diante (e o resto da turma, via `git pull`) vai usar — mesmo que o TSE atualize o arquivo-fonte depois. Só repita a Fase 0 e recommite quando quiser atualizar de propósito.

## Próximo passo

Vá para **`01_analise_descritiva.ipynb`** e use exatamente os mesmos valores de `CARGO`, `UF` e `ANO_ELEICAO` na célula de configuração, para carregar este mesmo arquivo.